# Use Shoonya API

### Ref:
1. Shoonya api doc: https://shoonya.com/api-documentation
2. Shoonya Python: https://github.com/Shoonya-Dev/ShoonyaApi-py

- download .whl file from "https://github.com/Shoonya-Dev/ShoonyaApi-py/blob/master/dist/NorenRestApi-0.0.30-py2.py3-none-any.whl"
- pip install tdf_utility/trading/shoonya/NorenRestApi-0.0.30-py2.py3-none-any.whl pyotp pandas

In [1]:
import os
import time
import json
import pandas as pd
from NorenRestApiPy.NorenApi import  NorenApi
import pyotp
# import logging

# logging.basicConfig(level=logging.DEBUG)

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global shoonya_api
        shoonya_api = self

shoonya_api = ShoonyaApiPy()

def shoonya_login() -> dict:
    #credentials
    uid     = os.getenv("SHOONYA_UID")
    pwd     = os.getenv("SHOONYA_PWD")
    factor2 = pyotp.TOTP(os.getenv("SHOONYA_TOTP_SECRET")).now()
    vc      = os.getenv("SHOONYA_VENDOR_CODE")
    app_key = os.getenv("SHOONYA_API_KEY")
    imei    = os.getenv("SHOONYA_IMEI")

    # Login
    shoonya_login_user = shoonya_api.login(userid=uid, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)
    return shoonya_login_user

def shoonya_watchlists(shoonya_login_user: dict) -> pd.DataFrame:
    watchlists_list = []
    for key in shoonya_login_user.get('mws'):
        watchlist_df = pd.DataFrame(shoonya_login_user.get('mws').get(key))
        watchlist_df["watchlist_name"] = key
        watchlists_list.append(watchlist_df)
    watchlists_df = pd.concat(watchlists_list).reset_index(drop=True)
    return watchlists_df

def shoonya_limits() -> dict:
    limits = shoonya_api.get_limits()
    return limits

def shoonya_search_scrip(serch_text: str, tism_search: str, exchange: str='NSE') -> str:
    search_scrip_data = shoonya_api.searchscrip(exchange=exchange, searchtext=serch_text)
    search_scrip_df = pd.DataFrame(search_scrip_data.get('values'))
    token = None
    if not search_scrip_df.loc[search_scrip_df['tsym'] == tism_search, "token"].empty:
        token = search_scrip_df.loc[search_scrip_df['tsym'] == tism_search, "token"].values[0]
    return token

def get_time(time_string):
    data = time.strptime(time_string,'%d-%m-%Y %H:%M:%S')
    return int(time.mktime(data))


shoonya_login_user = shoonya_login()
shoonya_limits_dict = shoonya_limits()
shoonya_watchlists_df = shoonya_watchlists(shoonya_login_user)
nifty50_token = shoonya_search_scrip(serch_text="NIFTY", tism_search="Nifty 50", exchange="NSE")


In [14]:
# ret = shoonya_login_user
# print(ret)
# print(f"Current Time: {ret.get('request_time')}")
# print(f"User Name: {ret.get('uname')}")
# print(f"User Email: {ret.get('email')}")
# print(f"User Mobile: {ret.get('m_num')}")
# print(f"User ID: {ret.get('uid')}")

# 'orarr': ['MKT', 'LMT', 'SL-LMT', 'SL-MKT']
# susertoken
# susertokenspl

In [16]:
# shoonya_limits_dict

In [3]:
shoonya_watchlists_df

,exch,token,tsym,cname,instname,pp,ls,ti,seg,watchlist_name,nontrd
0,NSE,16675,BAJAJFINSV-EQ,BAJAJ FINSERV LTD.,EQ,2,1,0.10,EQT,Nifty500,NaN
1,NSE,10999,MARUTI-EQ,MARUTI SUZUKI INDIA LTD.,EQ,2,1,1.00,EQT,Nifty500,NaN
2,NSE,759782,TMCV-EQ,TATA MOTORS LIMITED,EQ,2,1,0.05,EQT,Nifty500,NaN
3,NSE,3456,TMPV-EQ,TATA MOTORS PASS VEH LTD,EQ,2,1,0.05,EQT,Nifty500,NaN
4,NSE,10604,BHARTIARTL-EQ,BHARTI AIRTEL LIMITED,EQ,2,1,0.10,EQT,Nifty500,NaN
5,NSE,305,BAJAJHLDNG-EQ,BAJAJ HOLDINGS & INVS LTD,EQ,2,1,1.00,EQT,Nifty500,NaN
6,NSE,2885,RELIANCE-EQ,RELIANCE INDUSTRIES LTD,EQ,2,1,0.10,EQT,Nifty500,NaN
7,NSE,8479,TVSMOTOR-EQ,TVS MOTOR COMPANY LTD,EQ,2,1,0.10,EQT,Nifty500,NaN
8,NSE,3426,TATAPOWER-EQ,TATA POWER CO LTD,EQ,2,1,0.05,EQT,Nifty500,NaN
9,NSE,7229,HCLTECH-EQ,HCL TECHNOLOGIES LTD,EQ,2,1,0.10,EQT,Nifty500,NaN


In [53]:
print(f"Token: {nifty50_token}")

Token: 26000


In [54]:
nifty_option_scrip = shoonya_api.searchscrip(exchange='NFO', searchtext='Nifty')
nifty_option_scrip_df = pd.DataFrame(nifty_option_scrip.get('values'))
nifty_option_scrip_df


# nifty50_deriv = shoonya_search_scrip(serch_text="NIFTY", tism_search="Nifty 50", exchange="NFO")

,exch,token,tsym,dname,optt,instname,symname,seg,exd,pp,ls,ti
0,NFO,49234,NIFTYNXT5027JAN26F,NIFTYNXT50 27JAN FUT,XX,FUTIDX,NIFTYNXT50,DER,27-JAN-2026,2,25,0.20
1,NFO,49229,NIFTY27JAN26F,NIFTY JAN FUT,XX,FUTIDX,NIFTY,DER,27-JAN-2026,2,65,0.10
2,NFO,59187,NIFTYNXT5024FEB26F,NIFTYNXT50 24FEB FUT,XX,FUTIDX,NIFTYNXT50,DER,24-FEB-2026,2,25,0.20
3,NFO,59182,NIFTY24FEB26F,NIFTY FEB FUT,XX,FUTIDX,NIFTY,DER,24-FEB-2026,2,65,0.10
4,NFO,51715,NIFTYNXT5030MAR26F,NIFTYNXT50 30MAR FUT,XX,FUTIDX,NIFTYNXT50,DER,30-MAR-2026,2,25,0.20
5,NFO,51714,NIFTY30MAR26F,NIFTY MAR FUT,XX,FUTIDX,NIFTY,DER,30-MAR-2026,2,65,0.10
6,NFO,64640,NIFTY27JAN26C19800,NIFTY 27JAN26 19800 CE,CE,OPTIDX,NIFTY,DER,27-JAN-2026,2,65,0.05
7,NFO,64641,NIFTY27JAN26P19800,NIFTY 27JAN26 19800 PE,PE,OPTIDX,NIFTY,DER,27-JAN-2026,2,65,0.05
8,NFO,64650,NIFTY27JAN26C19850,NIFTY 27JAN26 19850 CE,CE,OPTIDX,NIFTY,DER,27-JAN-2026,2,65,0.05
9,NFO,64651,NIFTY27JAN26P19850,NIFTY 27JAN26 19850 PE,PE,OPTIDX,NIFTY,DER,27-JAN-2026,2,65,0.05


In [55]:
nifty_options_data = shoonya_api.get_security_info(exchange='NSE', token=nifty50_token)
nifty_options_data

{'request_time': '01:44:19 26-01-2026',
 'stat': 'Ok',
 'exch': 'NSE',
 'tsym': 'Nifty 50',
 'cname': 'NIFTY INDEX',
 'symname': 'NIFTY',
 'seg': 'EQT',
 'instname': 'UNDIND',
 'pp': '2',
 'prcftr': '1.000000',
 'ls': '1',
 'ti': '0.05',
 'mult': '1',
 'nontrd': '1',
 'prcftr_d': '(1 / 1 ) * (1 / 1)',
 'token': '26000'}

In [52]:
# watchlists_df
# search_scrip_df
nifty_options_data

In [ ]:
# Ref: https://www.youtube.com/watch?v=vaZMly5mkJw
# Strategy 1: Paiso Ka Ped in Nifty50 ETF
# Objective: To generate consistent returns by investing in high volume Nifty50 ETF using a systematic approach.
# ------------------------------------------------------------------------------------
# Define Parameters (Configurable)
# ------------------------------------------------------------------------------------
exchange = 'NSE'
nifty50_etf_ticker = 'NIFTYBEES'
weekly_sip_amount = 4000  # Amount to invest every week
weekly_sip_day = 'wednesday'  # Day of the week to invest
weekly_sip_start_time = "14:45"  # Start time for placing SIP order
weekly_sip_end_time = "15:15"    # End time for placing SIP order
profit_booking_percentage = 0.25  # 25% profit booking

# ------------------------------------------------------------------------------------
# Step 1: After Selecting High Volume Nifty50 ETF,
# ------------------------------------------------------------------------------------
nifty50_etf_scrip = shoonya_api.searchscrip(exchange=exchange, searchtext=nifty50_etf_ticker).get('values')[0]
nifty50_etf_token = nifty50_etf_scrip.get('token')
nifty50_etf_trading_symbol = nifty50_etf_scrip.get('tsym')

print("Step 1: Select High Volume Nifty50 ETF")
print(f'Nifty50 ETF: {nifty50_etf_ticker} & Token: {nifty50_etf_token}')
print()

# ------------------------------------------------------------------------------------
# Step 2: Analyze Strategy On Historical Data (Backtesting)
# ------------------------------------------------------------------------------------
# startdate = get_time("01-01-2024 09:15:00")
# enddate = get_time("01-01-2026 15:30:00")
# nifty50_daily_price_series = shoonya_api.get_daily_price_series(exchange = exchange,
#                                                                 tradingsymbol = nifty50_etf_trading_symbol,
#                                                                 startdate = startdate,
#                                                                 enddate = enddate)
# nifty50_daily_price_series_parsed = [json.loads(row) for row in nifty50_daily_price_series]
# nifty50_daily_price_series_df = pd.DataFrame(nifty50_daily_price_series_parsed)
# nifty50_daily_price_series_df.rename(columns={"time": "Date", "into": "Open", "inth": "High", "intl": "Low", "intc": "Close", "intv": "Volume"}, inplace=True)  # SSBOE stands for Seconds Since Begin Of Epoch
# nifty50_daily_price_series_df = nifty50_daily_price_series_df.loc[:, ["Date", "Open", "High", "Low", "Close", "Volume"]]
# nifty50_daily_price_series_df.sort_index(ascending=False, inplace=True)
# print("Step 2: Analyze Strategy On 2 Years Historical Data (Backtesting)")
# print(f"Historical Data: {nifty50_daily_price_series_df.shape}")
# print()
# ------------------------------------------------------------------------------------
# Step 3: Execute Strategy in Live Market
# ------------------------------------------------------------------------------------
# (Weekly SIP) Place order on every Wednesday/Friday from 2.45 PM to 3.15 PM based on close price of that day
# (Profit Booking) Place order to book profit of 20% of SIP Amount on every Monday/Thursday from 9.30 AM to 10.00 AM


Step 1: Select High Volume Nifty50 ETF
Nifty50 ETF: NIFTYBEES & Token: 10576

Step 2: Analyze 2 Years Historical Data
Historical Data: (490, 6)


In [90]:
nifty50_daily_price_series_df

,Date,Open,High,Low,Close,Volume
489,02-JAN-2024,241.00,241.00,238.00,239.21,2724450.00
488,03-JAN-2024,240.00,240.00,237.50,237.74,2253252.00
487,04-JAN-2024,237.70,239.59,237.19,239.30,2592068.00
486,05-JAN-2024,239.80,240.00,238.01,239.74,3017800.00
485,08-JAN-2024,242.00,242.00,237.50,237.68,3109332.00
...,...,...,...,...,...,...
4,26-DEC-2025,296.99,296.99,294.20,294.46,3502261.00
3,29-DEC-2025,294.46,295.28,292.76,293.15,5796571.00
2,30-DEC-2025,293.54,293.58,291.95,293.32,8060541.00
1,31-DEC-2025,294.08,295.70,293.06,295.31,5677730.00


In [6]:
shoonya_api.logout()

{'stat': 'Ok', 'request_time': '22:47:22 27-01-2026'}

# Nifty Data

# Testing

In [ ]:
def sourcing_viz_data():
    duration = "1Y"
    current_date = dt.today()
    year = str(current_date.year)
    from_dt = (current_date - td(days=365)).strftime("%d-%m-%Y")
    to_dt = (current_date).strftime("%d-%m-%Y")

    nse_api = NSE_API()
    
    nifty_50_historical_data = nse_api._get_data(f"api/NextApi/apiClient/historicalGraph?functionName=getIndexChart&&index=NIFTY%2050&flag={duration}")

    india_vix_historical_data = get_nse_india_vix()
    india_vix_historical_data['date'] = pd.to_datetime(india_vix_historical_data['date'], format='%d-%b-%Y')

    sleep(2)
    gold_historical_data = nse_api._get_data(f"api/historical-spot-price?symbol=GOLD&fromDate={from_dt}&toDate={to_dt}")
    gold_historical_df = pd.DataFrame(gold_historical_data['data']) \
                                        .loc[:, ['UpdatedDate', 'SpotPrice2']] \
                                        .rename(columns={'UpdatedDate': 'Date', 'SpotPrice2': 'Gold 10gm'})
    gold_historical_df['Date'] = pd.to_datetime(gold_historical_df['Date'], format='%d-%b-%Y')
    gold_historical_df['Gold 10gm'] = gold_historical_df['Gold 10gm'].astype(float)

    sleep(2)
    silver_historical_data = nse_api._get_data(f"api/historical-spot-price?symbol=SILVER&fromDate={from_dt}&toDate={to_dt}")
    silver_historical_df = pd.DataFrame(silver_historical_data['data']) \
                                        .loc[:, ['UpdatedDate', 'SpotPrice2']] \
                                        .rename(columns={'UpdatedDate': 'Date', 'SpotPrice2': 'Silver 1kg'})
    silver_historical_df['Date'] = pd.to_datetime(silver_historical_df['Date'], format='%d-%b-%Y')
    silver_historical_df['Silver 1kg'] = silver_historical_df['Silver 1kg'].astype(float)

    crudeoil_historical_data = nse_api._get_data(f"api/historical-spot-price?symbol=CRUDEOIL&fromDate={from_dt}&toDate={to_dt}")
    crudeoil_historical_df = pd.DataFrame(crudeoil_historical_data['data']) \
                                        .loc[:, ['UpdatedDate', 'SpotPrice1']] \
                                        .rename(columns={'UpdatedDate': 'Date', 'SpotPrice1': 'Crude Oil'})
    crudeoil_historical_df['Date'] = pd.to_datetime(crudeoil_historical_df['Date'], format='%d-%b-%Y')
    crudeoil_historical_df['Crude Oil'] = crudeoil_historical_df['Crude Oil'].astype(float)

    fii_dii_data_df = fetch_fii_dii_data()

    nifty50_historical_graph_df, nifty50_graph_identifier = _load_graph_data_to_df(nifty_50_historical_data)

    viz_df = pd.merge(nifty50_historical_graph_df.loc[:, ['Date', 'Price']].rename(columns={'Price': 'Nifty 50'}),
                india_vix_historical_data[['date', 'close']].rename(columns={'date': 'Date', 'close': 'India VIX'}),
                on='Date', how='left')
    viz_df = pd.merge(viz_df, gold_historical_df, on='Date', how='left')
    viz_df = pd.merge(viz_df, silver_historical_df, on='Date', how='left')
    viz_df = pd.merge(viz_df, crudeoil_historical_df, on='Date', how='left')
    viz_df = pd.merge(viz_df, fii_dii_data_df, on='Date', how='left')

    # market_status = get_nse_market_status_daily()
    # st.sidebar.write("Data is fetched and stored into cache")
    return viz_df